In [ ]:
#!/usr/bin/env python3
"""
Plot φ1 and φ2 probability-density distributions.

Each replica is converted into a separate probability-density
distribution. Mean and sample SD are then calculated across replicas.

Example:
    python plot_to_torsion_distributions.py \
        --input data/processed/torsion/torsion_na_antiparallel.npz \
        --output-prefix results/figures/torsion/to_na_antiparallel \
        --ion Na
"""

import argparse
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


ANGLE_LABELS = {
    "phi1": r"$\phi_1$ angle (degrees)",
    "phi2": r"$\phi_2$ angle (degrees)",
}


def parse_arguments():
    """讀取命令列參數。"""

    parser = argparse.ArgumentParser(
        description="Plot TO torsional probability densities."
    )

    parser.add_argument(
        "--input",
        required=True,
        type=Path,
        help="Input torsion NPZ file.",
    )

    parser.add_argument(
        "--output-prefix",
        required=True,
        type=Path,
        help="Output path without the phi1/phi2 suffix.",
    )

    parser.add_argument(
        "--ion",
        choices=["K", "Na"],
        required=True,
        help="Ion environment displayed in the legend.",
    )

    parser.add_argument(
        "--bins",
        type=int,
        default=36,
        help="Number of histogram bins. Default: 36.",
    )

    parser.add_argument(
        "--angle-range",
        nargs=2,
        type=float,
        default=(-90.0, 90.0),
        metavar=("MIN", "MAX"),
        help="Folded torsion range. Default: -90 90.",
    )

    parser.add_argument(
        "--dpi",
        type=int,
        default=300,
        help="PNG resolution. Default: 300.",
    )

    parser.add_argument(
        "--no-show",
        action="store_true",
        help="Save figures without displaying them.",
    )

    return parser.parse_args()


def find_system_prefixes(data):
    """自動尋找 sys_0、sys_1 等系統前綴。"""

    prefixes = []

    for key in data.files:
        match = re.fullmatch(
            r"(sys_\d+)_label",
            key,
        )

        if match:
            prefixes.append(
                match.group(1)
            )

    return sorted(
        set(prefixes),
        key=lambda value: int(
            value.split("_")[1]
        ),
    )


def fold_angles(angles):
    """將二面角折疊至 −90° 到 90°。"""

    return (
        np.asarray(angles, dtype=float)
        + 90.0
    ) % 180.0 - 90.0


def calculate_replica_density(
    angles,
    bins,
    angle_range,
):
    """計算單一 replica 的 probability density。"""

    folded_angles = fold_angles(
        angles
    )

    density, bin_edges = np.histogram(
        folded_angles,
        bins=bins,
        range=angle_range,
        density=True,
    )

    bin_centers = (
        bin_edges[:-1]
        + bin_edges[1:]
    ) / 2.0

    return bin_centers, density


def collect_density_statistics(
    data,
    prefix,
    angle_key,
    bins,
    angle_range,
):
    """計算單一系統跨 replicas 的 density Mean ± SD。"""

    replica_ids = np.asarray(
        data[f"{prefix}_replica_ids"],
        dtype=int,
    )

    replica_densities = []
    bin_centers = None

    for replica_id in replica_ids:
        key = (
            f"{prefix}_rep{replica_id}_{angle_key}"
        )

        if key not in data.files:
            print(f"[警告] 缺少：{key}")
            continue

        current_centers, density = (
            calculate_replica_density(
                angles=data[key],
                bins=bins,
                angle_range=angle_range,
            )
        )

        if bin_centers is None:
            bin_centers = current_centers

        replica_densities.append(
            density
        )

    if not replica_densities:
        return None

    stacked_density = np.vstack(
        replica_densities
    )

    replica_count = stacked_density.shape[0]

    mean_density = np.mean(
        stacked_density,
        axis=0,
    )

    std_density = np.std(
        stacked_density,
        axis=0,
        ddof=1 if replica_count > 1 else 0,
    )

    return {
        "centers": bin_centers,
        "mean": mean_density,
        "std": std_density,
        "replicas": replica_count,
    }


def load_all_statistics(data, prefixes, args):
    """預先計算 φ1、φ2 所有系統的分布統計。"""

    statistics = {
        "phi1": {},
        "phi2": {},
    }

    for angle_key in statistics:
        for prefix in prefixes:
            result = collect_density_statistics(
                data=data,
                prefix=prefix,
                angle_key=angle_key,
                bins=args.bins,
                angle_range=args.angle_range,
            )

            if result is not None:
                statistics[angle_key][prefix] = result

    return statistics


def find_global_y_limit(statistics):
    """取得 φ1、φ2 共用的 y 軸上限。"""

    maximum_density = 0.0

    for angle_statistics in statistics.values():
        for result in angle_statistics.values():
            upper_density = (
                result["mean"]
                + result["std"]
            )

            maximum_density = max(
                maximum_density,
                np.max(upper_density),
            )

    if maximum_density <= 0:
        raise RuntimeError(
            "無法取得有效 probability density。"
        )

    return maximum_density * 1.10


def plot_distribution(
    data,
    prefixes,
    statistics,
    angle_key,
    y_upper,
    args,
):
    """繪製單一 torsion 的 probability-density distribution。"""

    fig, ax = plt.subplots(
        figsize=(12, 8),
        dpi=150,
    )

    plot_count = 0

    for prefix in prefixes:
        if prefix not in statistics[angle_key]:
            continue

        result = statistics[angle_key][prefix]

        label = str(
            data[f"{prefix}_label"].item()
        )

        color = str(
            data[f"{prefix}_color"].item()
        )

        display_label = (
            f"{args.ion}⁺ {label}"
        )

        ax.plot(
            result["centers"],
            result["mean"],
            color=color,
            linewidth=2.8,
            alpha=1.0,
            label=display_label,
        )

        ax.fill_between(
            result["centers"],
            result["mean"] - result["std"],
            result["mean"] + result["std"],
            color=color,
            alpha=0.15,
            linewidth=0,
        )

        print(
            f"{angle_key}, {label}: "
            f"{result['replicas']} replicas"
        )

        plot_count += 1

    if plot_count == 0:
        plt.close(fig)
        raise RuntimeError(
            f"沒有有效的 {angle_key} 資料。"
        )

    ax.set_xlabel(
        ANGLE_LABELS[angle_key],
        fontsize=30,
    )

    ax.set_ylabel(
        "Probability density",
        fontsize=30,
    )

    ax.set_xlim(
        args.angle_range
    )

    ax.set_ylim(
        0,
        y_upper,
    )

    ax.set_xticks(
        [-90, -45, 0, 45, 90]
    )

    ax.tick_params(
        axis="both",
        which="major",
        labelsize=24,
    )

    ax.legend(
        fontsize=18,
        loc="upper right",
        frameon=True,
        framealpha=0.9,
    )

    ax.grid(
        True,
        linestyle="--",
        linewidth=0.8,
        alpha=0.4,
    )

    fig.tight_layout()

    png_path = Path(
        f"{args.output_prefix}_{angle_key}.png"
    )

    pdf_path = Path(
        f"{args.output_prefix}_{angle_key}.pdf"
    )

    png_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    fig.savefig(
        png_path,
        dpi=args.dpi,
        bbox_inches="tight",
    )

    fig.savefig(
        pdf_path,
        bbox_inches="tight",
    )

    print(f"Saved PNG: {png_path}")
    print(f"Saved PDF: {pdf_path}")

    return fig


def main():
    """主程式。"""

    args = parse_arguments()

    plt.rcParams["axes.linewidth"] = 1.8
    plt.rcParams["xtick.major.width"] = 1.8
    plt.rcParams["ytick.major.width"] = 1.8
    plt.rcParams["pdf.fonttype"] = 42
    plt.rcParams["ps.fonttype"] = 42

    if not args.input.exists():
        raise FileNotFoundError(
            f"找不到輸入檔案：{args.input}"
        )

    with np.load(
        args.input,
        allow_pickle=False,
    ) as data:
        prefixes = find_system_prefixes(
            data
        )

        if not prefixes:
            raise RuntimeError(
                "沒有找到 sys_0、sys_1 等系統資料。"
            )

        statistics = load_all_statistics(
            data=data,
            prefixes=prefixes,
            args=args,
        )

        y_upper = find_global_y_limit(
            statistics
        )

        figures = []

        for angle_key in ("phi1", "phi2"):
            figure = plot_distribution(
                data=data,
                prefixes=prefixes,
                statistics=statistics,
                angle_key=angle_key,
                y_upper=y_upper,
                args=args,
            )

            figures.append(figure)

        if args.no_show:
            for figure in figures:
                plt.close(figure)
        else:
            plt.show()


if __name__ == "__main__":
    main()